# Final Validation: SAE Diagnostics and Graph-Held-Out Tests

This notebook adds the two validation analyses that provide the greatest additional evidential value without retraining SAEs or regenerating attribution graphs. It measures SAE reconstruction/sparsity on the original splits, then tests whether the existing math and units graph-derived interventions generalise to new prompts.

Expected runtime is substantially shorter than graph construction. Activation recapture and diagnostics should take several minutes; the held-out intervention benchmark may take roughly 10--20 minutes depending on the Colab GPU. Run the cells in order.

## 1. Mount Drive and fetch the current repository


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = 'https://github.com/evey-dev/test_run.git'
repo_dir = '/content/test_run'

def run_cmd(command):
    print('$', ' '.join(command))
    subprocess.run(command, check=True)

github_ok = False
try:
    checkout = repo_dir
    if os.path.isdir(os.path.join(checkout, '.git')):
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        if os.path.exists(checkout) and os.listdir(checkout):
            checkout = '/content/test_run_github'
        if os.path.isdir(os.path.join(checkout, '.git')):
            run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
        elif os.path.exists(checkout) and os.listdir(checkout):
            raise RuntimeError(f'{checkout} exists but is not a git repository')
        else:
            run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
    os.chdir(checkout)
    github_ok = True
    print('Using GitHub checkout:', os.getcwd())
except Exception as exc:
    print('GitHub checkout failed; using Drive project.zip backup.')
    print(repr(exc))

if not github_ok:
    zip_path = '/content/drive/MyDrive/mphil-project/project.zip'
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Could not find {zip_path}')
    run_cmd(['unzip', '-q', '-o', zip_path, '-d', '/content/'])
    for candidate in ['/content/test_run', '/content/mphil_project/test_run', '/content']:
        if os.path.isdir(os.path.join(candidate, 'src')) and os.path.isdir(os.path.join(candidate, 'configs')):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError('Could not locate the extracted repository root')

print('Current working directory:', os.getcwd())


## 2. Install dependencies and regenerate deterministic prompt CSVs


In [ ]:
!pip install -q --upgrade "transformers>=4.51.0" accelerate matplotlib
!pip install -q -e .
!python data/generate_datasets.py --capitals


## 3. Restore existing final checkpoints and graph JSONs

No training or graph generation occurs in this notebook. The cell fails explicitly if a complete checkpoint set or either required graph is unavailable.

In [ ]:
from pathlib import Path
import shutil

layers = [4, 8, 12, 16, 20, 24, 28]
checkpoint_specs = {
    'math': {
        'local': Path('mechanistic_data/sae_checkpoints_math_final_train'),
        'drive': [
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_math_final_train'),
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_math_final_train'),
        ],
    },
    'units': {
        'local': Path('mechanistic_data/sae_checkpoints_units_final_train'),
        'drive': [
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_units_final_train'),
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_units_final_train'),
        ],
    },
    'capitals': {
        'local': Path('mechanistic_data/sae_checkpoints_capitals_final_train'),
        'drive': [
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/finals/sae_checkpoints_capitals_final_train'),
            Path('/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train'),
        ],
    },
}

def complete_checkpoint_set(path):
    return all(
        (path / f'sae_layer{layer}.pt').exists()
        and (path / f'sae_layer{layer}_metadata.json').exists()
        for layer in layers
    )

for domain, spec in checkpoint_specs.items():
    destination = spec['local']
    if complete_checkpoint_set(destination):
        print(f'{domain}: using existing local checkpoints')
        continue
    source = next((path for path in spec['drive'] if complete_checkpoint_set(path)), None)
    if source is None:
        raise FileNotFoundError(f'No complete {domain} checkpoint set found in: {spec["drive"]}')
    destination.mkdir(parents=True, exist_ok=True)
    for layer in layers:
        for suffix in ('.pt', '_metadata.json'):
            name = f'sae_layer{layer}{suffix}'
            shutil.copy2(source / name, destination / name)
    print(f'{domain}: restored checkpoints from {source}')

drive_outputs = Path('/content/drive/MyDrive/mphil-project/outputs')
local_outputs = Path('outputs')
local_outputs.mkdir(exist_ok=True)
required_graphs = [
    'math_final_carry_58_83_4v3_graph.json',
    'units_final_force_graph.json',
]
for name in required_graphs:
    destination = local_outputs / name
    if not destination.exists():
        source = drive_outputs / name
        if not source.exists():
            raise FileNotFoundError(f'Missing required graph: {source}')
        shutil.copy2(source, destination)
    print('Graph ready:', destination)


## 4. Recapture the original final-position activation bundles when needed

These are the same deterministic 1,000-prompt corpora and seed used for training. Recapture is required because the checkpoint directories contain weights and metadata, not the original activation matrices.

In [ ]:
import subprocess
import sys

capture_specs = {
    'math': ('addition', Path('mechanistic_data_math_final_train')),
    'units': ('units', Path('mechanistic_data_units_final_train')),
    'capitals': ('capitals', Path('mechanistic_data_capitals_final_train')),
}
for domain, (behaviour, output_dir) in capture_specs.items():
    complete = (output_dir / 'train_val_indices_per_layer.npy').exists() and all(
        (output_dir / f'activations_layer{layer}.npy').exists() for layer in layers
    )
    if complete:
        print(f'{domain}: activation bundle already present; skipping recapture')
        continue
    print(f'{domain}: capturing 1,000 final-position activation vectors per layer')
    subprocess.run([
        sys.executable, 'src/capture_activations.py',
        '--output-dir', str(output_dir),
        '--behaviours', behaviour,
        '--layers', *map(str, layers),
        '--seed', '787',
    ], check=True)


## 5. Measure reconstruction fidelity and achieved sparsity

Interpretation rule: poor validation FVE would support reconstruction insufficiency as one contributor to weak interventions. High FVE with weak sparse interventions would reject that simple explanation and instead implicate feature selection, sparsity, distribution mismatch or nonlinear interactions. Reconstruction metrics alone do not prove a causal explanation.

In [ ]:
diagnostic_configs = {
    'math': 'configs/sae_math_final_train_config.yaml',
    'units': 'configs/sae_units_final_train_config.yaml',
    'capitals': 'configs/sae_capitals_final_train_config.yaml',
}
for domain, config in diagnostic_configs.items():
    subprocess.run([
        sys.executable, 'src/sae_diagnostics.py',
        '--config', config,
        '--label', domain,
        '--output-json', f'outputs/final_sae_diagnostics_{domain}.json',
        '--output-csv', f'outputs/final_sae_diagnostics_{domain}.csv',
        '--device', 'auto',
    ], check=True)


In [ ]:
import pandas as pd
from IPython.display import display

diagnostics = pd.concat(
    [pd.read_csv(f'outputs/final_sae_diagnostics_{domain}.csv') for domain in diagnostic_configs],
    ignore_index=True,
)
display(diagnostics[[
    'label', 'layer', 'validation_fraction_variance_explained',
    'validation_mean_relative_l2_error', 'validation_mean_cosine_similarity',
    'validation_mean_l0', 'combined_dead_feature_fraction',
    'decoder_norm_p05', 'decoder_norm_median', 'decoder_norm_p95',
]].round(4))


## 6. Run graph-held-out arithmetic and units interventions

The arithmetic pairs are absent from the complete 1,000-prompt SAE corpus. The units source prompts are selected from the SAE validation split and the matched energy variants are absent from that corpus. Each condition uses literal edit strength 1.0. The existing graphs are reused; no graph is selected after observing these results.

In [ ]:
subprocess.run([
    sys.executable, 'src/heldout_validation.py',
    '--math-cases', '12',
    '--unit-cases', '12',
    '--output', 'outputs/final_heldout_validation.json',
], check=True)


In [ ]:
import json

with open('outputs/final_heldout_validation.json', 'r') as handle:
    heldout = json.load(handle)
for domain in ('math', 'units'):
    summary = heldout[domain]['summary']
    print(f'\n{domain.upper()}: {summary["eligible_cases"]}/{summary["total_cases"]} baseline-qualified')
    display(pd.DataFrame(summary['conditions']).T.round(4))


## 7. Generate report figures

Black diamonds denote condition means; individual points are retained so the report does not hide variation across prompts. The attribution-graph panel is explicitly a backward-connected visual subset of the complete JSON/HTML graph.

In [ ]:
subprocess.run([
    sys.executable, 'src/plot_validation.py',
    '--diagnostics',
    'outputs/final_sae_diagnostics_math.csv',
    'outputs/final_sae_diagnostics_units.csv',
    'outputs/final_sae_diagnostics_capitals.csv',
    '--heldout', 'outputs/final_heldout_validation.json',
    '--output-dir', 'outputs/report_figures',
], check=True)
subprocess.run([
    sys.executable, 'src/plot_attribution_graph.py',
    '--graph', 'outputs/math_final_carry_58_83_4v3_graph.json',
    '--output-dir', 'outputs/report_figures',
    '--stem', 'fig_math_attribution_graph',
], check=True)

from IPython.display import Image, display
display(Image(filename='outputs/report_figures/fig_sae_diagnostics.png'))
display(Image(filename='outputs/report_figures/fig_heldout_generalisation.png'))
display(Image(filename='outputs/report_figures/fig_math_attribution_graph.png'))


## 8. Persist new results and figures to Drive


In [ ]:
from glob import glob

drive_output = Path('/content/drive/MyDrive/mphil-project/outputs')
drive_figure_output = drive_output / 'report_figures'
drive_output.mkdir(parents=True, exist_ok=True)
drive_figure_output.mkdir(parents=True, exist_ok=True)

for pattern in ('outputs/final_sae_diagnostics_*', 'outputs/final_heldout_validation.json'):
    for source_name in glob(pattern):
        source = Path(source_name)
        shutil.copy2(source, drive_output / source.name)
        print('Copied', source.name)
for source_name in glob('outputs/report_figures/*'):
    source = Path(source_name)
    shutil.copy2(source, drive_figure_output / source.name)
    print('Copied figure', source.name)
